# Test FastCUT (bidirectional) on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/valerybr/contrastive-unpaired-translation/blob/master/colab_test_fastcut_bidirectional.ipynb)

Runs `test.py` for a **bidirectional shared-G** CUT/FastCUT checkpoint through the
deterministic *paired* `bilateral` adapter — the Colab equivalent of
`scripts/test-fastcut-bidirectional.sh`.

With `--bidirectional` the single shared generator is applied in **both** directions
on each study, so every saved study yields four aligned images:

| visual | meaning |
|---|---|
| `real_A` | L  (left CC, canonical-oriented domain) |
| `fake_B` | G(real_A)  (L translated to the right/B domain) |
| `real_B` | R  (paired right CC, flipped to L orientation) |
| `fake_A` | G(real_B)  (R translated back to the left/A domain) |

**Before running:** Runtime → Change runtime type → **GPU**.

### Expected Google Drive layout
Checkpoints and the VinDr-Mammo dataset live on Drive. Point the variables in
step 3 at *your* paths. The defaults assume:

```
MyDrive/phd/
├── data/vindr-masks/
│   ├── images/<study_id>/<image_id>.png         # + <image_id>_mask.png next to each
│   └── finding_annotations.csv
└── fastcut_checkpoints/
    └── vindr_bidirectional_ddp_g1_nce5_masks/
        └── latest_net_G.pth                       # (or <EPOCH>_net_G.pth)
```

## 1. Check GPU

In [1]:
!nvidia-smi

Mon Jun 15 08:23:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

Colab VMs are ephemeral; the checkpoint and dataset are read from Drive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Configure paths & run options

Edit these to match your Drive. The cell asserts that the dataset, annotations CSV
and checkpoint actually exist, so you get a clear error *now* rather than deep inside
the test run.

In [16]:
import os

# --- Google Drive inputs (checkpoints + dataset) ---
DRIVE_ROOT      = '/content'
DATAROOT        = f'{DRIVE_ROOT}/vindr/images'
ANNOTATIONS_CSV = f'{DRIVE_ROOT}/vindr/finding_annotations.csv'
CHECKPOINTS_DIR = f'{DRIVE_ROOT}/fastcut_checkpoints'

# --- run options (mirror scripts/test-fastcut-bidirectional.sh) ---
NAME           = 'vindr_bilateral_ddp_g1_nce5_m_bd_recon_l15_l20_20260614'
EPOCH          = 'latest'
SPLIT          = 'training'      # VinDr split column to load
NUM_TEST       = 200            # number of studies to translate
FINDING_FILTER = 'either_finding'  # no_finding | left_finding | right_finding | either_finding
BILATERAL_SIZE = '512 360'      # H W (must be multiples of 4)
CROP_WIDTH     = 360            # crop width after flip; 0 disables; <= BILATERAL_SIZE width

# --- results stay local for fast PNG writes; copy to Drive later if you want ---
RESULTS_DIR = '/content/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# --- fail fast if the Drive paths are wrong ---
ckpt = os.path.join(CHECKPOINTS_DIR, NAME, f'{EPOCH}_net_G.pth')
assert os.path.isdir(DATAROOT),         f'dataset dir not found: {DATAROOT}'
assert os.path.isfile(ANNOTATIONS_CSV), f'annotations csv not found: {ANNOTATIONS_CSV}'
assert os.path.isfile(ckpt),            f'checkpoint not found: {ckpt}'
print('OK  generator checkpoint ->', ckpt)

OK  generator checkpoint -> /content/fastcut_checkpoints/vindr_bilateral_ddp_g1_nce5_m_bd_recon_l15_l20_20260614/latest_net_G.pth


## 4. Clone the repo

In [6]:
%cd /content
![ -d contrastive-unpaired-translation ] || git clone https://github.com/valerybr/contrastive-unpaired-translation.git
%cd /content/contrastive-unpaired-translation

/content
Cloning into 'contrastive-unpaired-translation'...
remote: Enumerating objects: 563, done.
remote: Counting objects: 100% (267/267), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 563 (delta 172), reused 159 (delta 123), pack-reused 296 (from 2)
Receiving objects: 100% (563/563), 18.10 MiB | 20.58 MiB/s, done.
Resolving deltas: 100% (317/317), done.
/content/contrastive-unpaired-translation


## 5. Install dependencies

Colab already ships PyTorch + torchvision + OpenCV (`cv2`, used by the bilateral
loader); we only need the repo's lightweight extras.

In [7]:
!pip install -q dominate visdom GPUtil packaging

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


## 6. Run the bidirectional test

Loads `<checkpoints_dir>/<name>/<epoch>_net_G.pth` and writes the four-up images to
`<results_dir>/<name>/test_<epoch>/`. `--num_threads 2` matches Colab's 2 vCPUs.

In [17]:
cmd = f'''python test.py \
  --dataroot {DATAROOT} \
  --annotations_csv {ANNOTATIONS_CSV} \
  --dataset_mode bilateral \
  --finding_filter {FINDING_FILTER} \
  --name {NAME} \
  --model cut --CUT_mode FastCUT \
  --bidirectional True \
  --flip_right \
  --masked_loss True \
  --split {SPLIT} \
  --phase test \
  --epoch {EPOCH} \
  --num_test {NUM_TEST} \
  --results_dir {RESULTS_DIR} \
  --checkpoints_dir {CHECKPOINTS_DIR} \
  --crop_width {CROP_WIDTH} \
  --bilateral_size {BILATERAL_SIZE} \
  --num_threads 2'''
print(cmd)
!{cmd}

python test.py   --dataroot /content/vindr/images   --annotations_csv /content/vindr/finding_annotations.csv   --dataset_mode bilateral   --finding_filter either_finding   --name vindr_bilateral_ddp_g1_nce5_m_bd_recon_l15_l20_20260614   --model cut --CUT_mode FastCUT   --bidirectional True   --flip_right   --masked_loss True   --split training   --phase test   --epoch latest   --num_test 200   --results_dir /content/results   --checkpoints_dir /content/fastcut_checkpoints   --crop_width 360   --bilateral_size 512 360   --num_threads 2
----------------- Options ---------------
                 CUT_mode: FastCUT                       	[default: CUT]
          annotations_csv: /content/vindr/finding_annotations.csv	[default: None]
               batch_size: 1                             
            bidirectional: True                          	[default: False]
           bilateral_size: [512, 360]                    	[default: (512, 384)]
          checkpoints_dir: /content/fastcut_check

## 7. Write `findings.json` (bounding boxes)

Emits the finding boxes for `real_A` (L) and `real_B` (R) in the same flipped,
L-canonical, cropped frame as the saved PNGs, so they line up when overlaid in
step 9 (or in `util/overlay-bd.html`).

In [18]:
cmd = f'''python -m util.write_findings \
  --dataroot {DATAROOT} \
  --annotations_csv {ANNOTATIONS_CSV} \
  --finding_filter {FINDING_FILTER} \
  --name {NAME} \
  --split {SPLIT} \
  --phase test \
  --epoch {EPOCH} \
  --results_dir {RESULTS_DIR} \
  --crop_width {CROP_WIDTH} \
  --bilateral_size {BILATERAL_SIZE} \
  --flip_right'''
print(cmd)
!{cmd}

python -m util.write_findings   --dataroot /content/vindr/images   --annotations_csv /content/vindr/finding_annotations.csv   --finding_filter either_finding   --name vindr_bilateral_ddp_g1_nce5_m_bd_recon_l15_l20_20260614   --split training   --phase test   --epoch latest   --results_dir /content/results   --crop_width 360   --bilateral_size 512 360   --flip_right
[BilateralDataset] Loaded 629 paired studies (split=training) [finding_filter=either_finding]
[write_findings] wrote /content/results/vindr_bilateral_ddp_g1_nce5_m_bd_recon_l15_l20_20260614/test_latest/findings.json (629 images, 815 boxes)


## 8. Preview the four-up images

Each row is one study: `real_A` (L) · `fake_B` (G(L)) · `real_B` (R) · `fake_A` (G(R)).

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

N_SHOW = 6   # how many studies to preview

img_root = os.path.join(RESULTS_DIR, NAME, f'test_{EPOCH}', 'images')
labels = ['real_A', 'fake_B', 'real_B', 'fake_A']
names = sorted(os.path.basename(p) for p in glob.glob(os.path.join(img_root, 'real_A', '*.png')))
names = names[:N_SHOW]
assert names, f'no images found under {img_root}/real_A — did step 6 run?'

fig, axes = plt.subplots(len(names), 4, figsize=(14, 3.2 * len(names)))
axes = axes.reshape(len(names), 4)
for r, fn in enumerate(names):
    for c, lab in enumerate(labels):
        ax = axes[r][c]
        path = os.path.join(img_root, lab, fn)
        if os.path.isfile(path):
            ax.imshow(Image.open(path), cmap='gray')
        ax.set_axis_off()
        if r == 0:
            ax.set_title(lab)
    axes[r][0].text(-0.05, 0.5, fn[:18], rotation=90, va='center', ha='right',
                    transform=axes[r][0].transAxes, fontsize=8)
plt.tight_layout()
plt.show()

## 9. (Optional) Overlay finding boxes

Draws the `findings.json` boxes on `real_A` (L) and `real_B` (R) for studies that
carry a finding. The translated `fake_B` / `fake_A` are shown box-free for comparison.

In [12]:
# import json
# from PIL import Image, ImageDraw
# import matplotlib.pyplot as plt

# N_SHOW = 6
# web_dir = os.path.join(RESULTS_DIR, NAME, f'test_{EPOCH}')
# img_root = os.path.join(web_dir, 'images')
# with open(os.path.join(web_dir, 'findings.json')) as f:
#     findings = json.load(f)

# # pair each laterality's real with its translation; box only on the real
# panes = [('real_A', 'real_A'), ('fake_B', None), ('real_B', 'real_B'), ('fake_A', None)]
# names = [n for n in sorted(findings) if findings[n]][:N_SHOW]
# assert names, 'findings.json has no boxes — try FINDING_FILTER=either_finding and rerun steps 6-7'

# def load_boxed(label, box_key, fn):
#     img = Image.open(os.path.join(img_root, label, fn)).convert('RGB')
#     if box_key:
#         d = ImageDraw.Draw(img)
#         for b in findings[fn].get(box_key, []):
#             x0, y0, x1, y1 = b['box']
#             d.rectangle([x0, y0, x1, y1], outline=(255, 60, 60), width=3)
#     return img

# fig, axes = plt.subplots(len(names), 4, figsize=(14, 3.2 * len(names)))
# axes = axes.reshape(len(names), 4)
# for r, fn in enumerate(names):
#     for c, (label, box_key) in enumerate(panes):
#         ax = axes[r][c]
#         ax.imshow(load_boxed(label, box_key, fn))
#         ax.set_axis_off()
#         if r == 0:
#             ax.set_title(label + (' +box' if box_key else ''))
# plt.tight_layout()
# plt.show()

## 10. (Optional) Persist results to Drive

Results were written locally for speed. Copy the run to Drive to keep it.

In [15]:
DRIVE_RESULTS = f'/content/drive/MyDrive/phd/models/fastcut_results'
src = os.path.join(RESULTS_DIR, NAME, f'test_{EPOCH}')
dst = os.path.join(DRIVE_RESULTS, NAME, f'test_{EPOCH}')
os.makedirs(os.path.dirname(dst), exist_ok=True)
!cp -r '{src}' '{dst}'
print('copied ->', dst)

copied -> /content/drive/MyDrive/phd/models/fastcut_results/vindr_bilateral_ddp_g1_nce5_m_bd_recon_l10_l25_20260612/test_latest
